In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('credit_data.csv')
# the change in debt
for i in range(1, 6):
    df[f'BILL_DIFF_{i}'] = df[f'BILL_AMT{i}'] - df[f'BILL_AMT{i+1}']

# rolling windon 6 monthes
bill_cols = ['BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6']
pay_status_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
pay_amt_cols = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']

df['MEAN_BILL'] = df[bill_cols].mean(axis=1)
df['MAX_PAY_STATUS'] = df[pay_status_cols].max(axis=1)
df['SUM_PAY_AMT'] = df[pay_amt_cols].sum(axis=1)

# did they pay off last bill
df['PAY_RATIO_1'] = df['PAY_AMT1'] / (df['BILL_AMT2'] + 1)

# convert categorical variables for LGBM
cat_cols = ['SEX', 'EDUCATION', 'MARRIAGE']
for col in cat_cols:
    df[col] = df[col].astype('category')

print(f"New features created. Total columns: {len(df.columns)}")

New features created. Total columns: 34


In [2]:
import os
# Ensure the folders exist
os.makedirs('data', exist_ok=True)
os.makedirs('Models', exist_ok=True)

# Load from the new data folder
df = pd.read_csv('data/credit_data.csv')

In [ ]:
import lightgbm as lgb
import joblib
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_auc_score, classification_report


target = 'default payment next month'
X = df.drop(columns=[target, 'ID']) 
y = df[target]

# 80% Train/CV, 20% Final Test)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Save test set for evaluation
test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('data/credit_data_test.csv', index=False)

# Params for LGBM
params = {
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'seed': 42
}

# CV loop
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fold_aucs = []
best_iterations = []

print("Starting Cross-Validation...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train_full)):
    X_tr, X_val = X_train_full.iloc[train_idx], X_train_full.iloc[val_idx]
    y_tr, y_val = y_train_full.iloc[train_idx], y_train_full.iloc[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data, categorical_feature=cat_cols)
    
    model = lgb.train(
        params,
        train_data,
        valid_sets=[train_data, val_data],
        valid_names=['train', 'valid'],
        num_boost_round=1000,
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=0)]
    )
    
    y_pred = model.predict(X_val)
    auc_score = roc_auc_score(y_val, y_pred)
    fold_aucs.append(auc_score)
    best_iterations.append(model.best_iteration)
    print(f"Fold {fold+1} AUC: {auc_score:.4f}")

avg_auc = np.mean(fold_aucs)
avg_iteration = int(np.mean(best_iterations))
print(f"\nAverage CV AUC: {avg_auc:.4f}")

# retrain
print(f"Retraining final model on full training set with {avg_iteration} rounds...")
full_train_data = lgb.Dataset(X_train_full, label=y_train_full, categorical_feature=cat_cols)
final_model = lgb.train(params, full_train_data, num_boost_round=avg_iteration)

# eval 
test_preds = final_model.predict(X_test)
test_auc = roc_auc_score(y_test, test_preds)
print(f"Final Test Set AUC: {test_auc:.4f}")

# Save the model and features
final_model.save_model('Models/lgbm_credit_model.txt')
joblib.dump(X.columns.tolist(), 'Models/feature_columns.pkl')

print("\nModel and feature list saved successfully.")

Starting Cross-Validation...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[71]	train's auc: 0.844514	valid's auc: 0.785801
Fold 1 AUC: 0.7858
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[68]	train's auc: 0.840244	valid's auc: 0.787025
Fold 2 AUC: 0.7870
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[51]	train's auc: 0.83285	valid's auc: 0.774568
Fold 3 AUC: 0.7746
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[50]	train's auc: 0.829058	valid's auc: 0.788901
Fold 4 AUC: 0.7889
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[100]	train's auc: 0.855954	valid's auc: 0.788424
Fold 5 AUC: 0.7884

Average CV AUC: 0.7849
Retraining final model on full training set with 68 rounds...
Final Test Set AUC: 0.7773

Model and feature list saved successf